In [1]:
import os
import re
import sys
import glob

import pandas as pd
from bs4 import BeautifulSoup

try:
    from tqdm import tqdm
except ImportError:                      # tqdm optional
    def tqdm(x, **k):
        return x

In [14]:
MIN_CHARS = 1500          # below this, almost certainly a table-of-contents hit
FALLBACK_WINDOW = 120_000  # chars to take when no Item 1A is found

MIN_CHARS = 1500          # below this, almost certainly a table-of-contents hit
FALLBACK_WINDOW = 120_000  # chars to take when no Item 1A is found

# "Item 1. Business", "ITEM 1 - BUSINESS", "Item 1: Our Business", etc.
RE_START = re.compile(
    r'item\s*[\.\s]*1\s*[\.\:\-–—]?\s*(?:\u2014|\-)?\s*'
    r'(?:the\s+|our\s+)?business',
    re.I)

# "Item 1A. Risk Factors"
RE_END_1A = re.compile(
    r'item\s*[\.\s]*1\s*a\s*[\.\:\-–—]?\s*(?:risk|certain\s+risk)',
    re.I)

# fallback end marker if the filing has no Item 1A
RE_END_2 = re.compile(
    r'item\s*[\.\s]*2\s*[\.\:\-–—]?\s*(?:propert|description\s+of\s+propert)',
    re.I)


In [6]:
def html_to_text(path):
    """Strip markup, scripts, styles and tables; collapse whitespace."""
    with open(path, encoding='utf-8', errors='ignore') as fh:
        raw = fh.read()
    soup = BeautifulSoup(raw, 'lxml')
    # tables are financial data, not product vocabulary - they pollute the
    # word vectors and inflate character counts
    for tag in soup(['script', 'style', 'table']):
        tag.decompose()
    txt = soup.get_text(' ')
    txt = txt.replace('\xa0', ' ')
    return re.sub(r'\s+', ' ', txt).strip()

In [60]:
path = '../data/filings/2023/9728_314203_2023-12-31.html'

In [61]:
with open(path, encoding='utf-8', errors='ignore') as fh:
    raw = fh.read()
soup = BeautifulSoup(raw, 'lxml')
# tables are financial data, not product vocabulary - they pollute the
# word vectors and inflate character counts
for tag in soup(['script', 'style', 'table']):
        tag.decompose()
txt = soup.get_text(' ')
txt = txt.replace('\xa0', ' ')
txt = re.sub(r'\s+', ' ', txt).strip()

C:\Users\dheeraj.tommandru\AppData\Local\Temp\ipykernel_14644\3120825392.py:3: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(raw, 'lxml')


In [62]:
txt

'http://www.mcewenmining.com/20231231#LeaseLiabilityCurrent http://www.mcewenmining.com/20231231#LeaseLiabilityNonCurrent http://www.mcewenmining.com/20231231#LeaseLiabilityCurrent http://www.mcewenmining.com/20231231#LeaseLiabilityNonCurrent --12-31 2023 FY 0000314203 false http://www.mcewenmining.com/20231231#McewenCopperInc.Member 49440000 47428000 http://www.mcewenmining.com/20231231#LeaseLiabilityCurrent http://www.mcewenmining.com/20231231#LeaseLiabilityNonCurrent http://www.mcewenmining.com/20231231#LeaseLiabilityCurrent http://www.mcewenmining.com/20231231#LeaseLiabilityNonCurrent P2Y 0.50 0000314203 us-gaap:DebtSecuritiesMember 2022-01-01 2022-12-31 0000314203 mux:AmendedLoanAgreementMember 2022-03-31 2022-03-31 0000314203 mux:November2019OfferingMember mux:CommonStockAndAdditionalPaidInCapitalMember 2019-11-20 2019-11-20 0000314203 mux:McewenMember mux:StellantisPrivatePlacementMember 2023-10-11 2023-10-11 0000314203 mux:AmendedLoanAgreementMember 2020-06-25 2020-06-25 000031

In [22]:
def extract_item1(txt):
    """
    Return (body, status).

    Takes the LAST "Item 1 ... Business" heading that still has an Item 1A
    after it. The first such heading is nearly always the table of contents;
    the last is the real section.
    """
    starts = [m.start() for m in RE_START.finditer(txt)]
    if not starts:
        return None, 'no_item1_header'

    ends = [m.start() for m in RE_END_1A.finditer(txt)]
    if not ends:
        ends = [m.start() for m in RE_END_2.finditer(txt)]

    if ends:
        valid = [s for s in starts if any(e > s + 200 for e in ends)]
        if not valid:
            return None, 'no_end'
        s = max(valid)
        e = min(x for x in ends if x > s + 200)
    else:
        s = starts[-1]
        e = min(len(txt), s + FALLBACK_WINDOW)

    body = txt[s:e].strip()
    if len(body) < MIN_CHARS:
        return None, 'too_short'
    return body, 'ok'

In [63]:
item1 = extract_item1(txt)

In [64]:
item1

(None, 'no_item1_header')

In [47]:
txt

'Matson, Inc._December 31, 2025 http://xbrl.sec.gov/country/2025#US http://xbrl.sec.gov/country/2025#US 0000003453 2025 FY false http://xbrl.sec.gov/stpr/2025#CA http://xbrl.sec.gov/stpr/2025#HI http://xbrl.sec.gov/stpr/2025#CA http://xbrl.sec.gov/stpr/2025#HI 34 P2Y P95D http://fasb.org/us-gaap/2025#LiabilitiesCurrent http://xbrl.sec.gov/country/2025#US http://xbrl.sec.gov/stpr/2025#AK http://xbrl.sec.gov/stpr/2025#CA P6Y 0.80 0000003453 us-gaap:CommonStockMember 2025-01-01 2025-12-31 0000003453 us-gaap:CommonStockMember 2024-01-01 2024-12-31 0000003453 us-gaap:CommonStockMember 2023-01-01 2023-12-31 0000003453 us-gaap:RetainedEarningsMember 2025-12-31 0000003453 us-gaap:AdditionalPaidInCapitalMember 2025-12-31 0000003453 us-gaap:AccumulatedOtherComprehensiveIncomeMember 2025-12-31 0000003453 us-gaap:AccumulatedDefinedBenefitPlansAdjustmentMember 2025-12-31 0000003453 matx:AccumulatedOtherMember 2025-12-31 0000003453 matx:AccumulatedNonQualifiedPlansAdjustmentAttributableToParentMembe

In [67]:
starts = [m.start() for m in RE_START.finditer(txt)]

ends = [m.start() for m in RE_END_1A.finditer(txt)]
if not ends:
    ends = [m.start() for m in RE_END_2.finditer(txt)]

TypeError: 'callable_iterator' object is not subscriptable

In [51]:
ends

[76856, 137679]

In [34]:
from lxml import etree, html

In [52]:
# 1. Parse the document
with open(path, "r", encoding="utf-8") as html_file:
    tree = html.parse(html_file)

# 2. Get the root element (usually <html>)
root = tree.getroot()
print(f"Root Tag: {root.tag}")

# 3. Get all immediate child elements of the root
# In standard HTML files, this will typically be <head> and <body>
children = root.getchildren()
print(f"Number of immediate children under root: {len(children)}")

Root Tag: html
Number of immediate children under root: 2


In [39]:
with open(path, "r", encoding="utf-8") as html_file:
        # The HTML parser handles the broken or unclosed tags common in SEC filings
        tree = html.parse(html_file)

In [45]:
all_text = tree.xpath("string()")
all_text

'Matson,\xa0Inc._December\xa031, 2025http://xbrl.sec.gov/country/2025#UShttp://xbrl.sec.gov/country/2025#US00000034532025FYfalsehttp://xbrl.sec.gov/stpr/2025#CA http://xbrl.sec.gov/stpr/2025#HIhttp://xbrl.sec.gov/stpr/2025#CA http://xbrl.sec.gov/stpr/2025#HI34P2YP95Dhttp://fasb.org/us-gaap/2025#LiabilitiesCurrenthttp://xbrl.sec.gov/country/2025#UShttp://xbrl.sec.gov/stpr/2025#AK http://xbrl.sec.gov/stpr/2025#CAP6Y0.800000003453us-gaap:CommonStockMember2025-01-012025-12-310000003453us-gaap:CommonStockMember2024-01-012024-12-310000003453us-gaap:CommonStockMember2023-01-012023-12-310000003453us-gaap:RetainedEarningsMember2025-12-310000003453us-gaap:AdditionalPaidInCapitalMember2025-12-310000003453us-gaap:AccumulatedOtherComprehensiveIncomeMember2025-12-310000003453us-gaap:AccumulatedDefinedBenefitPlansAdjustmentMember2025-12-310000003453matx:AccumulatedOtherMember2025-12-310000003453matx:AccumulatedNonQualifiedPlansAdjustmentAttributableToParentMember2025-12-310000003453matx:AccumulatedDe

In [46]:
clean_text = " ".join(all_text.split())
clean_text

'Matson, Inc._December 31, 2025http://xbrl.sec.gov/country/2025#UShttp://xbrl.sec.gov/country/2025#US00000034532025FYfalsehttp://xbrl.sec.gov/stpr/2025#CA http://xbrl.sec.gov/stpr/2025#HIhttp://xbrl.sec.gov/stpr/2025#CA http://xbrl.sec.gov/stpr/2025#HI34P2YP95Dhttp://fasb.org/us-gaap/2025#LiabilitiesCurrenthttp://xbrl.sec.gov/country/2025#UShttp://xbrl.sec.gov/stpr/2025#AK http://xbrl.sec.gov/stpr/2025#CAP6Y0.800000003453us-gaap:CommonStockMember2025-01-012025-12-310000003453us-gaap:CommonStockMember2024-01-012024-12-310000003453us-gaap:CommonStockMember2023-01-012023-12-310000003453us-gaap:RetainedEarningsMember2025-12-310000003453us-gaap:AdditionalPaidInCapitalMember2025-12-310000003453us-gaap:AccumulatedOtherComprehensiveIncomeMember2025-12-310000003453us-gaap:AccumulatedDefinedBenefitPlansAdjustmentMember2025-12-310000003453matx:AccumulatedOtherMember2025-12-310000003453matx:AccumulatedNonQualifiedPlansAdjustmentAttributableToParentMember2025-12-310000003453matx:AccumulatedDefinedB

In [ ]:
xml_bytes = etree.tostring(parsed_html, method="xml", pretty_print=True)
xml_bytes

In [ ]:
xml_string = xml_bytes.decode("utf-8")
xml_string